In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2

In [208]:
from unsloth import FastLanguageModel 
import torch 

device = "cuda" if torch.cuda.is_available() else "cpu" 
max_seq_length = 1024 

model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct"  , 
    max_seq_length = max_seq_length , 
    dtype = torch.float16 , 
    load_in_4bit = True
)


==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:45: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


### Model Test

In [209]:
## testing up the model 
from unsloth.chat_templates  import get_chat_template 

FastLanguageModel.for_inference(model)  

messages = [
     {
        "role": "system",
        "content": "You are a SQL generator. Only output SQL query. No explanation, no markdown."
    },
    {
    "role":"user" , "content":"write the sql query to retrieve std_name and student roll number the table name student"
}]

def generate_response(messages): 
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    inputs = tokenizer(
        text , 
        return_tensors = "pt"
    ).to("cuda")
    outputs = model.generate(**inputs , max_new_tokens=64 , temperature=0.01) 

    return tokenizer.decode(outputs[0]) 
print(generate_response(messages))

<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Apr 2026

You are a SQL generator. Only output SQL query. No explanation, no markdown.<|eot_id|><|start_header_id|>user<|end_header_id|>

write the sql query to retrieve std_name and student roll number the table name student<|eot_id|><|start_header_id|>assistant<|end_header_id|>

SELECT std_name, roll_number FROM student<|eot_id|>


- we add the LoRA adapters so we only need to update 1 to 10% of parameters. rather doing full fine tunning. 

In [210]:
model = FastLanguageModel.get_peft_model(
    model , 
    r=16 , 
    target_modules=["q_proj" , "k_proj" , "v_proj" , "o_proj" , 
                    "gate_proj" , "up_proj" , "down_proj"] , 
    lora_alpha=16 , 
    lora_dropout = 0 , 
    bias = "none" , 
    use_gradient_checkpointing = "unsloth" , 
    random_state = 3407 , 
    use_rslora = False , 
    loftq_config = None
)

In [211]:
## loading the dataset 
from datasets import load_dataset 

ds = load_dataset("gretelai/synthetic_text_to_sql" , split="train") 
ds

Dataset({
    features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
    num_rows: 100000
})

In [212]:
ds[1]

{'id': 5098,
 'domain': 'defense industry',
 'domain_description': 'Defense contract data, military equipment maintenance, threat intelligence metrics, and veteran employment stats.',
 'sql_complexity': 'aggregation',
 'sql_complexity_description': 'aggregation functions (COUNT, SUM, AVG, MIN, MAX, etc.), and HAVING clause',
 'sql_task_type': 'analytics and reporting',
 'sql_task_type_description': 'generating reports, dashboards, and analytical insights',
 'sql_prompt': 'List all the unique equipment types and their corresponding total maintenance frequency from the equipment_maintenance table.',
 'sql_context': 'CREATE TABLE equipment_maintenance (equipment_type VARCHAR(255), maintenance_frequency INT);',
 'sql': 'SELECT equipment_type, SUM(maintenance_frequency) AS total_maintenance_frequency FROM equipment_maintenance GROUP BY equipment_type;',
 'sql_explanation': 'This query groups the equipment_maintenance table by equipment_type and calculates the sum of maintenance_frequency fo

In [213]:
def format_dataset(example):
    messages = [
        {
            "role": "system",
            "content": "You are a SQL generator. Only output SQL query. No explanation."
        },
        {
            "role": "user",
            "content": f"""Generate SQL query using the given schema.

Schema:
{example['sql_context']}

Question:
{example['sql_prompt']}"""
        },
        {
            "role": "assistant",
            "content": example["sql"].strip()
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": str(text)}   # ✅ MUST be string (not list)

In [214]:
example = format_dataset(ds[1]) 
example

{'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 07 Apr 2026\n\nYou are a SQL generator. Only output SQL query. No explanation.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nGenerate SQL query using the given schema.\n\nSchema:\nCREATE TABLE equipment_maintenance (equipment_type VARCHAR(255), maintenance_frequency INT);\n\nQuestion:\nList all the unique equipment types and their corresponding total maintenance frequency from the equipment_maintenance table.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nSELECT equipment_type, SUM(maintenance_frequency) AS total_maintenance_frequency FROM equipment_maintenance GROUP BY equipment_type;<|eot_id|>'}

In [215]:
print(example['text'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Apr 2026

You are a SQL generator. Only output SQL query. No explanation.<|eot_id|><|start_header_id|>user<|end_header_id|>

Generate SQL query using the given schema.

Schema:
CREATE TABLE equipment_maintenance (equipment_type VARCHAR(255), maintenance_frequency INT);

Question:
List all the unique equipment types and their corresponding total maintenance frequency from the equipment_maintenance table.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

SELECT equipment_type, SUM(maintenance_frequency) AS total_maintenance_frequency FROM equipment_maintenance GROUP BY equipment_type;<|eot_id|>


In [216]:
dataset = ds.map(format_dataset)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [217]:
dataset.column_names

['id',
 'domain',
 'domain_description',
 'sql_complexity',
 'sql_complexity_description',
 'sql_task_type',
 'sql_task_type_description',
 'sql_prompt',
 'sql_context',
 'sql',
 'sql_explanation',
 'text']

In [218]:
datasets = dataset.select_columns("text")

In [201]:
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [225]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = datasets,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 1000,
        learning_rate = 2e-4,
        logging_steps = 100,
        optim = "adamw_torch",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [226]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
4.576 GB of memory reserved.


In [227]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
100,0.555500
200,0.493200
300,0.483900
400,0.484600
500,0.466000
600,0.465500
700,0.454500
800,0.449900
900,0.448200
1000,0.446700


In [228]:
SAVE_DIR = "llama3_2_1b_text_to_sql"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

('llama3_2_1b_text_to_sql/tokenizer_config.json',
 'llama3_2_1b_text_to_sql/special_tokens_map.json',
 'llama3_2_1b_text_to_sql/chat_template.jinja',
 'llama3_2_1b_text_to_sql/tokenizer.json')

In [ ]:
from huggingface_hub import login 
import os  

token = os.getenv("your token")
# # print(token)
login(token)
model.push_to_hub("Rohit-Katkar2003/llama3_2_1b_text_to_sql")
tokenizer.push_to_hub("Rohit-Katkar2003/llama3_2_1b_text_to_sql")

### Fine-tune Model testing

In [ ]:
from unsloth import FastLanguageModel 

model1 , tokenizer1 = FastLanguageModel.from_pretrained("Rohit-Katkar2003/llama3_2_1b_text_to_sql")

FastLanguageModel.for_inference(model1)  

messages = [
     {
        "role": "system",
        "content": "You are a SQL generator. Only output SQL query. No explanation, no markdown."
    },
    {
    "role":"user" , "content":"""How many electric vehicles are there in China and Japan?.
    Schema: CREATE TABLE electric_vehicles (country VARCHAR(50), num_vehicles INT); 
    INSERT INTO electric_vehicles (country, num_vehicles) VALUES ('China', 1140000), ('Japan', 850000);"""
}]

def generate_response(messages): 
    text = tokenizer1.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer1(
        text , 
        return_tensors = "pt"
    ).to("cuda")
    outputs = model.generate(**inputs , max_new_tokens=64 , temperature=0.01) 

    return tokenizer1.decode(outputs[0]) 
print(generate_response(messages))

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:45: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Apr 2026

You are a SQL generator. Only output SQL query. No explanation, no markdown.<|eot_id|><|start_header_id|>user<|end_header_id|>

How many electric vehicles are there in China and Japan?.
    Schema: CREATE TABLE electric_vehicles (country VARCHAR(50), num_vehicles INT); 
    INSERT INTO electric_vehicles (country, num_vehicles) VALUES ('China', 1140000), ('Japan', 850000);<|eot_id|><|start_header_id|>assistant<|end_header_id|>

SELECT country, num_vehicles FROM electric_vehicles WHERE country IN ('China', 'Japan');<|eot_id|>


### Quantization

In [ ]:
# Merge to 16bit 
TOKEN = ""
if True: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if True: model.push_to_hub_merged("Rohit-Katkar2003/llama3_2_1b_text_to_sql_16bit", tokenizer, save_method = "merged_16bit", token = TOKEN)

# Merge to 4bit
if True: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if True: model.push_to_hub_merged("Rohit-Katkar2003/llama3_2_1b_text_to_sql_4bit", tokenizer, save_method = "merged_4bit_forced", token = TOKEN)

# # Just LoRA adapters
# if False:
#     model.save_pretrained("llama_lora")
#     tokenizer.save_pretrained("llama_lora")
# if False:
#     model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
#     tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:12<00:00, 12.47s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:17<00:00, 17.43s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/llama_finetune_16bit`
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:12<00:00, 12.27s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:49<00:00, 49.94s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/Rohit-Katkar2003/llama3_2_1b_text_to_sql_16bit`


RuntimeError: Unsloth: Merging into 4bit will cause your model to lose accuracy if you plan
to merge to GGUF or others later on. I suggest you to do this as a final step
if you're planning to do multiple saves.
If you are certain, change `save_method` to `merged_4bit_forced`.

- might try `merged_4bit_forced` work for 4bit 

In [238]:
# Save to 8bit Q8_0
if True: model.save_pretrained_gguf("llama3.2-1b-text-2-sql", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if True: model.push_to_hub_gguf("Rohit-Katkar2003/llama3.2-1b-text-2-sql", tokenizer, token =TOKEN)

# Save to 16bit GGUF
if True: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if True: model.push_to_hub_gguf("Rohit-Katkar2003/llama3.2-1b-text-2-sql", tokenizer, quantization_method = "f16", token = TOKEN)

# Save to q4_k_m GGUF
if True: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if True: model.push_to_hub_gguf("Rohit-Katkar2003/llama3.2-1b-text-2-sql", tokenizer, quantization_method = "q4_k_m", token = TOKEN)

# Save to multiple GGUF options - much faster if you want multiple!
if True:
    model.push_to_hub_gguf(
        "Rohit-Katkar2003/llama3.2-1b-text-2-sql", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = TOKEN, 
    )

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:10<00:00, 10.88s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:16<00:00, 16.57s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/llama3.2-1b-text-2-sql`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:17<00:00, 17.39s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf___ypiicj`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m', 'q8_0', 'q5_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf___ypiicj_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: [2] Converting GGUF f16 into q8_0. This might take 10 minutes...
Unsloth: [2] Converting GGUF f16 into q5_k_m. This might take 10 minutes...
Unsloth: Model

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Rohit-Katkar2003/llama3.2-1b-text-2-sql
Unsloth: Cleaning up temporary files...
